# Proyecto LSP — Fase 2: glosas → frase → voz (demo offline)

Demuestra el pipeline del video, sin webcam todavía:

**Seña real → reconocedor (top-5) → LLM arma la frase → TTS la habla.**

Usa el modelo entrenado en el notebook 03 (`modelo_pucp305_final.keras`, ~81% top-5)
y los arrays de test cacheados — **no descarga los 3.5 GB de MediaPipe**.

**Pre-requisitos:**
1. Haber corrido el notebook 03 (deja en `Drive/MyDrive/PUCP305_models/`: el modelo,
   `classes_pucp305.json` y `cache/X_test_f30.npy`, `cache/y_test_f30.npy`).
2. Una API key gratis de Gemini → https://aistudio.google.com/apikey . Agrégala como
   **secreto de Colab** (icono de llave 🔑) con el nombre `GEMINI_API_KEY`.

## Setup — repo, dependencias y Drive

In [ ]:
%cd /content
!rm -rf /content/Proyecto_Percepcion
!git clone -q https://github.com/Jtarazona00/Proyecto_Percepcion.git /content/Proyecto_Percepcion
%cd /content/Proyecto_Percepcion

import os, sys
os.environ['DATASET'] = 'pucp305'
sys.path.insert(0, '/content/Proyecto_Percepcion')

!pip install -q tensorflow==2.18.0 google-generativeai gTTS

from google.colab import drive
drive.mount('/content/drive')

## Paso 1 — Cargar modelo, clases y test cacheado

In [ ]:
import json
import numpy as np
import tensorflow as tf
from pathlib import Path
import config

MODELS_DRIVE = Path('/content/drive/MyDrive/PUCP305_models')
CACHE_DRIVE  = MODELS_DRIVE / 'cache'

model = tf.keras.models.load_model(MODELS_DRIVE / 'modelo_pucp305_final.keras', compile=False)
with open(MODELS_DRIVE / 'classes_pucp305.json', encoding='utf-8') as f:
    clases = json.load(f)          # lista; indice = NEW_ID

X_test = np.load(CACHE_DRIVE / f'X_test_f{config.FRAMES}.npy')
y_test = np.load(CACHE_DRIVE / f'y_test_f{config.FRAMES}.npy')
print(f'Modelo cargado. {len(clases)} clases. Test {X_test.shape}.')

## Paso 2 — API key de Gemini

Toma la key del secreto de Colab `GEMINI_API_KEY`. Si no lo configuraste, la pide por teclado.

In [ ]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    import getpass
    GEMINI_API_KEY = getpass.getpass('GEMINI_API_KEY: ')

from src.inferencia.llm_frase import listar_modelos
print('Modelos Gemini disponibles para tu key (primeros 5):')
for m in listar_modelos(GEMINI_API_KEY)[:5]:
    print('  ', m)
# Si 'gemini-1.5-flash' no aparece, edita MODELO_DEFECTO en src/inferencia/llm_frase.py

## Paso 3 — Elegir señas y reconocer (top-5)

Para cada glosa objetivo se toma una muestra REAL del set de test, se pasa por el
modelo y se obtienen sus 5 candidatos. Así se ve que aunque el top-1 falle, la seña
correcta suele estar en el top-5 — que es lo que recibe el LLM.

In [ ]:
# Secuencia de glosas que simula lo que señaría un paciente (deben existir como clases).
GLOSAS_OBJETIVO = ['DOLOR-DE-CABEZA', 'FIEBRE2', 'MAREO']

def muestra_test_de(label):
    cid = clases.index(label)
    idxs = np.where(y_test == cid)[0]
    return X_test[idxs[0]] if len(idxs) else None

def top5_labels(sample):
    p = model.predict(sample[None], verbose=0)[0]
    return [clases[i] for i in np.argsort(p)[-5:][::-1]]

secuencia = []
for g in GLOSAS_OBJETIVO:
    s = muestra_test_de(g)
    assert s is not None, f'La glosa {g!r} no existe como clase.'
    cand = top5_labels(s)
    acierto = 'OK' if g == cand[0] else ('top5' if g in cand else 'FALLA')
    print(f'seña real: {g:20} -> top-5: {cand}   [{acierto}]')
    secuencia.append(cand)

## Paso 4 — El LLM arma la frase

Gemini recibe la secuencia de candidatos top-5 y elige la combinación más coherente
para producir una frase natural en español (contexto hospitalario).

In [ ]:
from src.inferencia.llm_frase import armar_frase

frase = armar_frase(secuencia, api_key=GEMINI_API_KEY, contexto='hospitalario')
print('Glosas:', GLOSAS_OBJETIVO)
print('Frase :', frase)

## Paso 5 — TTS: la frase en voz

gTTS genera un mp3 (en Colab no funciona pyttsx3). Se reproduce abajo.

In [ ]:
from src.inferencia.tts import sintetizar_archivo
from IPython.display import Audio

ruta = sintetizar_archivo(frase, 'frase.mp3', lang='es', tld='com.mx')
Audio(ruta, autoplay=True)